
Graph-State Encrypted Cloning Certification (GSECC)
and Graph-State Decoder Construction (GSDC)
===========================================================================

This module implements the complete computational pipeline for the exact
certification of graph-state encrypted cloning resources and the
construction of the corresponding decoder.

The implementation follows the algorithms described in the accompanying
README and consists of six logical components.

---------------------------------------------------------------------------
1. GF(2) LINEAR ALGEBRA
---------------------------------------------------------------------------

Functions

    gf2_rank()
    gf2_is_invertible()

These routines implement Gaussian elimination over the binary field GF(2)
and are used to determine the rank and invertibility of the cut matrix

$$
\Gamma_{S,N}.
$$

The rank computation is the computational core of the GSECC certification
algorithm.

---------------------------------------------------------------------------
2. GRAPH UTILITIES
---------------------------------------------------------------------------

Functions

    adjacency_matrix()
    cut_matrix()

These utilities construct graph adjacency matrices and extract the cut
matrix associated with a balanced bipartition

$$
V=S\cup N.
$$

---------------------------------------------------------------------------
3. PARAMETER VALIDATION
---------------------------------------------------------------------------

Functions

    validate_graph_size()
    print_parameters()

These routines verify that the supplied graph satisfies

$$
|V|=2mk,
$$

where

- m : number of clones,
- k : number of signal qubits,
- mk : subsystem size.

Invalid parameter choices immediately raise a ValueError.

---------------------------------------------------------------------------
4. GSECC
---------------------------------------------------------------------------

Functions

    verify_certificate()
    find_certificate_exact()

verify_certificate()

    Polynomial-time verification of a candidate certificate.

    Returns True iff

   $$
\text{rank}_{GF(2)}(\Gamma_{S,N}) = mk.
$$

find_certificate_exact()

    Implements the exact certification algorithm.

    The routine

        • enumerates every balanced partition,
        • constructs Γ(S,N),
        • computes its GF(2) rank,
        • returns the first valid certificate.

    Returns

        (S,N)

    if a certificate exists.

    Returns

        None

    only when every balanced partition has been exhausted.

    Therefore, None constitutes a rigorous proof that no valid
    encrypted-cloning partition exists.

---------------------------------------------------------------------------
5. GRAPH-STATE CONSTRUCTION
---------------------------------------------------------------------------

Functions

    graph_state_from_adjacency()
    ghz_state()

    bell_pair_matching_graph()
    star_graph()
    path_graph()
    grid_graph()

These routines generate graph states and benchmark graph families used
throughout the demonstrations.

---------------------------------------------------------------------------
6. REDUCED DENSITY MATRICES
---------------------------------------------------------------------------

Functions

    reduced_density_matrix()
    rho_S_matches_maximally_mixed()

These routines compute

$$
\rho_S
$$

and verify whether

$$
\rho_S=\frac{I}{2^{mk}}
$$

within numerical precision.

Only the residual norm is reported.

The density matrix itself is not printed.

---------------------------------------------------------------------------
7. GSDC
---------------------------------------------------------------------------

Functions

    construct_decoder()

Given a certified partition

$$
(S,N),
$$

constructs the decoder unitary

$$
W
$$

satisfying

$$
|G\rangle
=
(I_S\otimes W)
|\Phi_{2^{mk}}\rangle.
$$

The routine additionally verifies

    • unitarity,

    • state reconstruction,

    • numerical residuals.

---------------------------------------------------------------------------
8. VISUALIZATION
---------------------------------------------------------------------------

Function

    plot_graph()

Generates publication-quality graph visualizations.

Certified graphs display

    • signal subsystem,

    • noise subsystem,

    • cut edges,

using separate colours.

Figures are automatically saved as

    • PDF

    • 600 dpi PNG

---------------------------------------------------------------------------
9. REPORTING
---------------------------------------------------------------------------

Functions

    report_certificate()

Runs the complete certification pipeline

    GF(2) verification
        ↓
    reduced density matrix
        ↓
    decoder construction
        ↓
    graph visualization

report_non_graph_state()

Applies the GSDC verification pipeline to arbitrary quantum states such
as GHZ states.

---------------------------------------------------------------------------
10. DEMONSTRATION
---------------------------------------------------------------------------

Executing the benchmark block evaluates the graph families using
the same names as in the manuscript table:

    • Path / linear cluster
    • Cycle C₂₄
    • Gᶜˡ₂×₁₂
    • Fixed G(24,0.35) sample
    • Fixed G(24,0.9) sample
    • Complete graph K₂₄
    • Star / GHZ graph

For every certified graph the script automatically

    • identifies the certificate,

    • verifies maximal mixedness,

    • constructs the decoder,

    • generates publication-quality figures,

    • reports numerical residuals.

For the complete mathematical formulation, proofs, computational
complexity, and theoretical background, see README.md.
---------------------------------------------------------------------------
11. REVISED EXACT GSECC (ADDED)
---------------------------------------------------------------------------

The original full-rank GSECC routines are retained for compatibility.

The notebook now additionally provides

    gf2_nullspace_basis()
    sector_partitions_exact()
    verify_gsecc_realization()
    find_gsecc_certificate_exact()
    report_gsecc_result()

These implement the revised exact condition

$$
\ker(B_{S,\mathcal N}^{\mathsf T})
\subseteq
\mathcal E_{P,Q}^{(m,k)}(A_S),
$$

including the odd-m rank-deficient exceptional branches and exhaustive
search over oriented balanced cuts and logical-sector decompositions.



In [35]:

from __future__ import annotations
import itertools
import os
import random
from typing import Iterable, Optional, Sequence, Tuple

import numpy as np

import matplotlib
matplotlib.use("Agg")  # headless / script-safe backend, no plt.show() needed
import matplotlib.pyplot as plt
import networkx as nx

try:
    FIGURE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    FIGURE_DIR = os.getcwd()

# PRX-friendly plotting defaults
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif", "STIXGeneral"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})



In [36]:
# GF(2) linear algebra

def gf2_rank(mat: np.ndarray) -> int:
    A = mat.copy().astype(np.uint8) % 2
    rows, cols = A.shape
    rank = 0
    for col in range(cols):
        pivot = None
        for r in range(rank, rows):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        A[[rank, pivot]] = A[[pivot, rank]]
        for r in range(rows):
            if r != rank and A[r, col]:
                A[r, :] ^= A[rank, :]
        rank += 1
        if rank == rows:
            break
    return rank


def gf2_is_invertible(mat: np.ndarray) -> bool:
    n, m = mat.shape
    if n != m:
        raise ValueError("Matrix must be square to test invertibility.")
    return gf2_rank(mat) == n


# Graph representation & cut matrix

def adjacency_matrix(n: int, edges: Iterable[Tuple[int, int]]) -> np.ndarray:
    A = np.zeros((n, n), dtype=np.uint8)
    for u, v in edges:
        A[u, v] = 1
        A[v, u] = 1
    return A


def cut_matrix(A: np.ndarray, S: Sequence[int], N: Sequence[int]) -> np.ndarray:
    return A[np.ix_(S, N)]



In [37]:
# (m, k) parameter validation

def validate_graph_size(A: np.ndarray, m: int, k: int) -> int:
    if m <= 0 or k <= 0:
        raise ValueError(f"m and k must be positive integers (got m={m}, k={k}).")
    mk = m * k
    n = A.shape[0]
    if n != 2 * mk:
        raise ValueError(
            f"Graph has {n} vertices, but expected 2*m*k = {2 * mk} "
            f"for m={m}, k={k}."
        )
    return mk


def print_parameters(m: int, k: int, A: np.ndarray) -> None:
    mk = m * k
    n = A.shape[0]
    print("Encrypted Cloning Parameters")
    print("----------------------------")
    print(f"m  = {m}")
    print(f"k  = {k}")
    print(f"mk = {mk}")
    print(f"Graph vertices = {n}")


# GSECC: verification of a single candidate certificate (S, N)

def verify_certificate(A: np.ndarray, S: Sequence[int], N: Sequence[int], m: int, k: int) -> bool:
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]
    if len(S) != len(N) or len(S) + len(N) != n or len(S) != mk:
        raise ValueError(f"S, N must each have size mk = {mk} and partition all {n} vertices.")
    if set(S) & set(N):
        raise ValueError("S and N must be disjoint.")
    Gamma = cut_matrix(A, list(S), list(N))
    return gf2_is_invertible(Gamma)


# GSECC: exhaustive search (exact, exponential -- fine for small n)

def find_certificate_exact(A: np.ndarray, m: int, k: int) -> Optional[Tuple[Tuple[int, ...], Tuple[int, ...]]]:
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]
    vertices = range(n)
    others = [v for v in vertices if v != 0]
    for combo in itertools.combinations(others, mk - 1):
        S = (0,) + combo
        N = tuple(v for v in vertices if v not in S)
        if verify_certificate(A, S, N, m, k):
            return S, N
    return None


### New exact GSECC algorithm: complete cut-kernel criterion

The original `verify_certificate()` / `find_certificate_exact()` routines are kept unchanged as the
**full-cut-rank shortcut**.  The functions below add the exact graph-state criterion used in the
revised manuscript.

For a fixed oriented balanced realization \(S:\mathcal N\), let

\[
A_S=A[S,S],\qquad B=A[S,\mathcal N],\qquad
\mathcal K=\ker_{\mathbb F_2}(B^{\mathsf T}).
\]

The exact test is

\[
\mathcal K\subseteq\mathcal E_{P,Q}^{(m,k)}(A_S).
\]

The implementation follows the parity/encoder branches directly:

- **even \(m\)**: full cut rank is necessary and sufficient;
- **odd \(m\), \(XY/YX\)**: full cut rank is necessary and sufficient;
- **odd \(m\), \(XZ/ZX\)**: every kernel basis vector must be sector-constant,
  \(w=Fc\), satisfy \((A_S+I)w=0\), and have even sector parity;
- **odd \(m\), \(YZ/ZY\)**: every kernel basis vector must be sector-constant,
  \(w=Fc\), and satisfy \(A_Sw=0\).

For the two exceptional odd-\(m\) branches, the graph-level exact search enumerates every
**oriented** balanced cut and, only when the cut is rank deficient but not ruled out by the
necessary rank bound, every inequivalent partition of \(S\) into \(k\) sectors of size \(m\).
Sector permutations are removed by a canonical recursive enumeration.


In [38]:
# ============================================================================
# NEW EXACT GSECC ALGORITHM
# Complete cut-kernel criterion with parity- and encoder-dependent branches
# ============================================================================

def gf2_nullspace_basis(mat: np.ndarray) -> np.ndarray:
    """
    Return a row-wise basis of ker(mat) over GF(2).

    If mat has shape (r, c), the returned array has shape (nullity, c).
    """
    A = np.asarray(mat, dtype=np.uint8).copy() % 2
    rows, cols = A.shape

    pivot_cols = []
    pivot_row = 0

    # Reduced row-echelon form over GF(2)
    for col in range(cols):
        pivot = None
        for r in range(pivot_row, rows):
            if A[r, col]:
                pivot = r
                break

        if pivot is None:
            continue

        A[[pivot_row, pivot]] = A[[pivot, pivot_row]]

        for r in range(rows):
            if r != pivot_row and A[r, col]:
                A[r, :] ^= A[pivot_row, :]

        pivot_cols.append(col)
        pivot_row += 1

        if pivot_row == rows:
            break

    free_cols = [c for c in range(cols) if c not in pivot_cols]
    if not free_cols:
        return np.zeros((0, cols), dtype=np.uint8)

    basis = []
    for free in free_cols:
        x = np.zeros(cols, dtype=np.uint8)
        x[free] = 1

        # In RREF: x_pivot + sum_j A[row,j] x_j = 0.
        # Over GF(2), subtraction equals addition.
        for row, pivot_col in reversed(list(enumerate(pivot_cols))):
            x[pivot_col] = np.dot(A[row, :], x) % 2

        basis.append(x)

    return np.asarray(basis, dtype=np.uint8)


def _normalize_encoder_class(encoder) -> str:
    """
    Normalize an ordered two-Pauli encoder to one of:
        'XZ/ZX', 'YZ/ZY', 'XY/YX'.

    Accepted examples:
        'XZ', 'ZX', ('X','Z'), ['Y','Z'], 'XZ/ZX'.
    """
    if isinstance(encoder, str):
        raw = encoder.upper().replace(" ", "").replace(",", "")
        if "/" in raw:
            parts = [p for p in raw.split("/") if p]
            if not parts:
                raise ValueError("Empty encoder specification.")
            raw = parts[0]
        raw = "".join(ch for ch in raw if ch in "XYZ")
        if len(raw) != 2:
            raise ValueError(
                "Encoder must specify two distinct Pauli axes, e.g. 'XZ', 'YZ', or 'XY'."
            )
        P, Q = raw[0], raw[1]
    else:
        if len(encoder) != 2:
            raise ValueError("Encoder must contain exactly two Pauli axes.")
        P, Q = str(encoder[0]).upper(), str(encoder[1]).upper()

    if P not in "XYZ" or Q not in "XYZ" or P == Q:
        raise ValueError(f"Invalid two-Pauli encoder ({P}, {Q}).")

    axes = frozenset((P, Q))
    if axes == frozenset(("X", "Z")):
        return "XZ/ZX"
    if axes == frozenset(("Y", "Z")):
        return "YZ/ZY"
    if axes == frozenset(("X", "Y")):
        return "XY/YX"

    raise ValueError(f"Unsupported encoder ({P}, {Q}).")


def _validate_balanced_cut(
    A: np.ndarray,
    S: Sequence[int],
    N: Sequence[int],
    m: int,
    k: int,
) -> int:
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]

    S = tuple(S)
    N = tuple(N)

    if len(S) != mk or len(N) != mk:
        raise ValueError(f"S and N must each have size mk = {mk}.")
    if len(set(S)) != mk or len(set(N)) != mk:
        raise ValueError("S and N cannot contain repeated vertices.")
    if set(S) & set(N):
        raise ValueError("S and N must be disjoint.")
    if set(S) | set(N) != set(range(n)):
        raise ValueError("S and N must partition all graph vertices.")

    return mk


def _validate_sectors(
    S: Sequence[int],
    sectors: Sequence[Sequence[int]],
    m: int,
    k: int,
) -> Tuple[Tuple[int, ...], ...]:
    sectors = tuple(tuple(sec) for sec in sectors)

    if len(sectors) != k:
        raise ValueError(f"Expected k = {k} logical sectors, got {len(sectors)}.")
    if any(len(sec) != m for sec in sectors):
        raise ValueError(f"Every logical sector must contain exactly m = {m} signal vertices.")

    flattened = [v for sec in sectors for v in sec]
    if len(set(flattened)) != m * k:
        raise ValueError("Logical sectors must be pairwise disjoint.")
    if set(flattened) != set(S):
        raise ValueError("Logical sectors must partition the complete signal set S.")

    return sectors


def sector_partitions_exact(
    S: Sequence[int],
    m: int,
    k: int,
):
    """
    Enumerate inequivalent partitions of S into k unlabeled sectors of size m.

    The smallest remaining signal vertex is forced into the next sector.
    This removes the k! redundancy from permuting logical-sector labels and
    yields exactly

        (mk)! / ((m!)^k k!)

    partitions.
    """
    S = tuple(sorted(S))
    if len(S) != m * k:
        raise ValueError(f"|S| must equal m*k = {m*k}.")

    def rec(remaining, blocks_left):
        remaining = tuple(remaining)

        if blocks_left == 0:
            if not remaining:
                yield tuple()
            return

        if len(remaining) != blocks_left * m:
            return

        anchor = remaining[0]
        rest = remaining[1:]

        for tail in itertools.combinations(rest, m - 1):
            block = tuple(sorted((anchor,) + tail))
            block_set = set(block)
            next_remaining = tuple(v for v in remaining if v not in block_set)

            for suffix in rec(next_remaining, blocks_left - 1):
                yield (block,) + suffix

    yield from rec(S, k)


def _sector_coefficients(
    w: np.ndarray,
    S: Sequence[int],
    sectors: Sequence[Sequence[int]],
) -> Optional[np.ndarray]:
    """
    If w is constant on every logical sector, return c with w = F c.
    Otherwise return None.

    The vector w is written in the row ordering of S.
    """
    pos = {v: idx for idx, v in enumerate(S)}
    coeffs = []

    for sec in sectors:
        vals = [int(w[pos[v]]) for v in sec]
        if any(bit != vals[0] for bit in vals):
            return None
        coeffs.append(vals[0])

    return np.asarray(coeffs, dtype=np.uint8)


def verify_gsecc_realization(
    A: np.ndarray,
    S: Sequence[int],
    N: Sequence[int],
    m: int,
    k: int,
    encoder="XZ",
    sectors: Optional[Sequence[Sequence[int]]] = None,
) -> dict:
    """
    Exact fixed-realization GSECC test.

    Tests
        ker(B^T) subset E_{P,Q}^{(m,k)}(A_S)

    using a GF(2) basis of ker(B^T).  Since the exceptional subspace is
    linear, testing a basis is sufficient.

    For rank-deficient odd-m XZ/ZX and YZ/ZY branches, `sectors` is required.
    Full-rank realizations do not require a sector decomposition.
    """
    mk = _validate_balanced_cut(A, S, N, m, k)
    S = tuple(S)
    N = tuple(N)
    encoder_class = _normalize_encoder_class(encoder)

    A_S = A[np.ix_(S, S)].astype(np.uint8) % 2
    B = cut_matrix(A, S, N).astype(np.uint8) % 2

    rank = gf2_rank(B)
    kernel_basis = gf2_nullspace_basis(B.T)
    nullity = kernel_basis.shape[0]

    info = {
        "signal": S,
        "noise": N,
        "rank": rank,
        "nullity": nullity,
        "status": "invalid",
        "valid": False,
        "branch": None,
        "encoder_class": encoder_class,
        "sectors": None,
        "kernel_basis": [tuple(int(x) for x in w) for w in kernel_basis],
        "reason": None,
    }

    # Full-rank branch: valid for every parity and every two-Pauli class.
    if nullity == 0:
        info.update(
            status="valid",
            valid=True,
            branch="full_rank",
            reason="ker(B^T) is trivial."
        )
        return info

    # Even m: exceptional space is trivial.
    if m % 2 == 0:
        info.update(
            branch="even_m_full_rank_required",
            reason="For even m, every nonzero cut-kernel vector is obstructive."
        )
        return info

    # Odd m, XY/YX: exceptional space is also trivial.
    if encoder_class == "XY/YX":
        info.update(
            branch="odd_XY_full_rank_required",
            reason="For odd m with XY/YX, the exceptional subspace is trivial."
        )
        return info

    # Necessary rank/nullity pruning before any sector search.
    if encoder_class == "XZ/ZX":
        min_rank = mk - k + 1
        max_nullity = k - 1
    else:  # YZ/ZY
        min_rank = (m - 1) * k
        max_nullity = k

    if rank < min_rank or nullity > max_nullity:
        info.update(
            branch="rank_bound_failed",
            reason=(
                f"Necessary rank bound failed for {encoder_class}: "
                f"rank={rank}, required rank >= {min_rank}."
            )
        )
        return info

    if sectors is None:
        info.update(
            status="needs_sectors",
            branch="exceptional_sector_test_required",
            reason=(
                "This rank-deficient odd-m branch requires a logical-sector "
                "decomposition of S."
            )
        )
        return info

    sectors = _validate_sectors(S, sectors, m, k)
    info["sectors"] = sectors

    I = np.eye(mk, dtype=np.uint8)

    for a, w in enumerate(kernel_basis):
        c = _sector_coefficients(w, S, sectors)

        if c is None:
            info.update(
                branch="exceptional_kernel_failed",
                reason=(
                    f"Kernel basis vector {a} is not constant within every "
                    "logical sector, so w != F c."
                )
            )
            return info

        if encoder_class == "XZ/ZX":
            lhs = ((A_S ^ I) @ w) % 2   # A_S + I over GF(2)
            if np.any(lhs):
                info.update(
                    branch="exceptional_kernel_failed",
                    reason=(
                        f"Kernel basis vector {a} fails (A_S + I) w = 0 "
                        "for the XZ/ZX class."
                    )
                )
                return info

            # Explicitly check the even-sector-parity condition.
            # For a simple graph and odd m it follows from (A_S+I)w=0,
            # but keeping it here mirrors the theorem exactly.
            if int(np.sum(c) % 2) != 0:
                info.update(
                    branch="exceptional_kernel_failed",
                    reason=(
                        f"Kernel basis vector {a} has odd sector parity "
                        "1_k^T c != 0 in the XZ/ZX class."
                    )
                )
                return info

        elif encoder_class == "YZ/ZY":
            lhs = (A_S @ w) % 2
            if np.any(lhs):
                info.update(
                    branch="exceptional_kernel_failed",
                    reason=(
                        f"Kernel basis vector {a} fails A_S w = 0 "
                        "for the YZ/ZY class."
                    )
                )
                return info

    info.update(
        status="valid",
        valid=True,
        branch="rank_deficient_exceptional",
        reason=(
            "Every basis vector of ker(B^T) lies in the encoder-selected "
            "exceptional subspace."
        )
    )
    return info


def find_gsecc_certificate_exact(
    A: np.ndarray,
    m: int,
    k: int,
    encoder="XZ",
) -> dict:
    """
    Exact graph-level GSECC search.

    Searches every ORIENTED balanced cut S:N.  In the exceptional odd-m
    branches, a rank-deficient candidate cut is followed by an exhaustive
    search over inequivalent sector partitions of S.

    Returns a dictionary with status='valid' and a complete certificate as
    soon as one is found.  Returns status='invalid' only after exhausting
    every admissible realization.

    Warning:
        The search is combinatorial.  For the exceptional odd-m branches,
        the worst-case enumeration contains

            C(2mk, mk) * (mk)! / ((m!)^k k!)

        fixed-realization tests, before graph-symmetry reductions.
    """
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]
    encoder_class = _normalize_encoder_class(encoder)

    cuts_checked = 0
    sector_partitions_checked = 0

    for S in itertools.combinations(range(n), mk):
        S = tuple(S)
        Sset = set(S)
        N = tuple(v for v in range(n) if v not in Sset)
        cuts_checked += 1

        B = cut_matrix(A, S, N)
        rank = gf2_rank(B)
        nullity = mk - rank

        # Full cut rank is universally valid, independent of sector grouping.
        if nullity == 0:
            result = verify_gsecc_realization(
                A, S, N, m, k, encoder=encoder_class, sectors=None
            )
            result["cuts_checked"] = cuts_checked
            result["sector_partitions_checked"] = sector_partitions_checked
            return result

        # Trivial-exceptional branches cannot accept rank deficiency.
        if m % 2 == 0 or encoder_class == "XY/YX":
            continue

        # Fast necessary rank/nullity pruning.
        if encoder_class == "XZ/ZX":
            if rank < mk - k + 1:
                continue
        elif encoder_class == "YZ/ZY":
            if rank < (m - 1) * k:
                continue

        # Exact sector search for the two exceptional odd-m branches.
        for sectors in sector_partitions_exact(S, m, k):
            sector_partitions_checked += 1
            result = verify_gsecc_realization(
                A, S, N, m, k, encoder=encoder_class, sectors=sectors
            )
            if result["valid"]:
                result["cuts_checked"] = cuts_checked
                result["sector_partitions_checked"] = sector_partitions_checked
                return result

    return {
        "status": "invalid",
        "valid": False,
        "signal": None,
        "noise": None,
        "rank": None,
        "nullity": None,
        "branch": "exhaustive_search_failed",
        "encoder_class": encoder_class,
        "sectors": None,
        "kernel_basis": [],
        "reason": (
            "Every oriented balanced cut and every required sector "
            "decomposition was exhausted."
        ),
        "cuts_checked": cuts_checked,
        "sector_partitions_checked": sector_partitions_checked,
    }


def report_gsecc_result(label: str, result: dict) -> None:
    """Compact reporter for the new exact GSECC certificate format."""
    print(f"{label}")
    print("-" * len(label))
    print(f"status        : {result['status']}")
    print(f"encoder class : {result['encoder_class']}")
    print(f"branch         : {result['branch']}")

    if result.get("signal") is not None:
        print(f"S              : {result['signal']}")
        print(f"N              : {result['noise']}")
        print(f"rank(B)        : {result['rank']}")
        print(f"nullity        : {result['nullity']}")
        print(f"kernel basis   : {result['kernel_basis']}")
        print(f"sectors        : {result['sectors']}")

    if "cuts_checked" in result:
        print(f"cuts checked   : {result['cuts_checked']}")
        print(f"sector checks  : {result['sector_partitions_checked']}")

    print(f"reason         : {result['reason']}")
    print()


In [39]:

# Exact graph-state / GHZ / Bell-pair construction

def graph_state_from_adjacency(A: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
    psi = plus
    for _ in range(n - 1):
        psi = np.kron(psi, plus)
    psi = psi.reshape([2] * n)
    for i in range(n):
        for j in range(i + 1, n):
            if A[i, j]:
                for bits in itertools.product([0, 1], repeat=n):
                    if bits[i] == 1 and bits[j] == 1:
                        psi[bits] *= -1
    return psi.reshape(-1)


def ghz_state(n: int) -> np.ndarray:
    psi = np.zeros(2 ** n, dtype=complex)
    psi[0] = 1 / np.sqrt(2)
    psi[-1] = 1 / np.sqrt(2)
    return psi


def bell_pair_matching_graph(m: int, k: int) -> np.ndarray:
    mk = m * k
    edges = [(i, i + mk) for i in range(mk)]
    return adjacency_matrix(2 * mk, edges)


def star_graph(m: int, k: int) -> np.ndarray:
    mk = m * k
    n = 2 * mk
    edges = [(0, v) for v in range(1, n)]
    return adjacency_matrix(n, edges)


def path_graph(m: int, k: int) -> np.ndarray:
    mk = m * k
    n = 2 * mk
    edges = [(i, i + 1) for i in range(n - 1)]
    return adjacency_matrix(n, edges)


def grid_graph(rows: int, cols: int) -> np.ndarray:
    n = rows * cols
    edges = []
    for r in range(rows):
        for c in range(cols):
            v = r * cols + c
            if c + 1 < cols:
                edges.append((v, v + 1))
            if r + 1 < rows:
                edges.append((v, v + cols))
    return adjacency_matrix(n, edges)


def reduced_density_matrix(state: np.ndarray, keep: Sequence[int], n: int) -> np.ndarray:
    keep = list(sorted(keep))
    trace = [i for i in range(n) if i not in keep]
    psi = state.reshape([2] * n)
    perm = keep + trace
    psi = np.transpose(psi, perm)
    dk = 2 ** len(keep)
    dt = 2 ** len(trace)
    psi = psi.reshape(dk, dt)
    return psi @ psi.conj().T


def rho_S_matches_maximally_mixed(rho_S: np.ndarray, m: int, k: int, atol: float = 1e-9):
    mk = m * k
    target = np.eye(2 ** mk) / (2 ** mk)
    residual = np.linalg.norm(rho_S - target)
    return residual <= atol, residual


In [40]:

# Visualization

def plot_graph(A: np.ndarray, S: Optional[Sequence[int]] = None, N: Optional[Sequence[int]] = None, title: str = "Graph", save: bool = True, fig_dir: str = FIGURE_DIR,) -> Optional[str]:
    
    G = nx.from_numpy_array(A)

    # PRX Quantum friendly palette
    SIGNAL_COLOR = "#56B4E9"      # sky blue
    NOISE_COLOR = "#009E73"       # PRX-style green
    CUT_EDGE_COLOR = "#4C5D73"    # muted blue-gray
    INSIDE_EDGE_COLOR = "0.75"
    DEFAULT_NODE_COLOR = "#56B4E9"

    plt.figure(figsize=(8, 7))

    if S is None or N is None:
        pos = nx.spring_layout(G, seed=5)
        nx.draw_networkx(
            G,
            pos,
            node_color=DEFAULT_NODE_COLOR,
            node_size=900,
            edgecolors="black",
            linewidths=1.2,
            font_size=12,
            font_weight="bold",
            font_family="serif",
        )
    else:
        S = list(S)
        N = list(N)
        mk = len(S)

        pos = {}
        y = list(range(mk))[::-1]

        for i, v in enumerate(S):
            pos[v] = (0, y[i])
        for i, v in enumerate(N):
            pos[v] = (4, y[i])

        labels_signal = {}
        labels_noise = {}
        
        # Signal vertices
        for idx, v in enumerate(S):
            j = idx // m + 1
            i = idx % m + 1
            labels_signal[v] = rf"$S_{{{j},{i}}}$"
        
        # Noise vertices
        for idx, v in enumerate(N):
            j = idx // m + 1
            i = idx % m + 1
            labels_noise[v] = rf"$N_{{{j},{i}}}$"
        
        Sset = set(S)
        Nset = set(N)
        cut = []
        inside = []
        for u, v in G.edges():
            if (u in Sset and v in Nset) or (u in Nset and v in Sset):
                cut.append((u, v))
            else:
                inside.append((u, v))

        nx.draw_networkx_edges(G, pos, edgelist=inside, edge_color=INSIDE_EDGE_COLOR, width=1.5)
        nx.draw_networkx_edges(G, pos, edgelist=cut, edge_color=CUT_EDGE_COLOR, width=3)

        nx.draw_networkx_nodes(
            G, pos, nodelist=S, node_color=SIGNAL_COLOR,
            edgecolors="black", linewidths=1.5, node_size=1200,
        )
        nx.draw_networkx_nodes(
            G, pos, nodelist=N, node_color=NOISE_COLOR,
            edgecolors="black", linewidths=1.5, node_size=1200,
        )

        # Put the indexed qubit labels inside the nodes.
        # Use a dark text colour for good contrast on both the sky-blue
        # and green fills.
        LABEL_COLOR = "#111111"

        combined_labels = {}
        combined_labels.update(labels_signal)
        combined_labels.update(labels_noise)

        nx.draw_networkx_labels(
            G, pos, combined_labels,
            font_size=14, font_weight="bold",
            font_color=LABEL_COLOR, font_family="serif"
        )

        plt.text(
            0, mk + 0.3, r"$S$",
            fontsize=18, ha="center",
            color=SIGNAL_COLOR, family="serif"
        )
        plt.text(
            4, mk + 0.3, r"$N$",
            fontsize=18, ha="center",
            color=NOISE_COLOR, family="serif"
        )

    plt.title(title, fontsize=18, family="serif")
    plt.axis("off")
    plt.tight_layout()

    pdf_path = None
    if save:
        os.makedirs(fig_dir, exist_ok=True)
        filename = title
        for bad, good in [(" ", "_"), ("(", ""), (")", ""), ("/", "_"),
                           ("\\", "_"), (",", ""), ("=", ""), (":", "")]:
            filename = filename.replace(bad, good)
        pdf_path = os.path.join(fig_dir, filename + ".pdf")
        png_path = os.path.join(fig_dir, filename + ".png")
        plt.savefig(pdf_path, bbox_inches="tight")
        plt.savefig(png_path, dpi=600, bbox_inches="tight")

    plt.close()
    return pdf_path


In [41]:
# GSDC: decoder construction

def _coefficient_matrix(psi: np.ndarray, S: Sequence[int], N: Sequence[int], n: int, mk: int) -> np.ndarray:
    S = list(S)
    N = list(N)
    perm = S + N
    psi_t = np.transpose(psi.reshape([2] * n), perm)
    return psi_t.reshape(2 ** mk, 2 ** mk)


def construct_decoder(psi: np.ndarray, S: Sequence[int], N: Sequence[int], m: int, k: int, atol: float = 1e-9):
    mk = m * k
    n = len(S) + len(N)
    d = 2 ** mk
    M = _coefficient_matrix(psi, S, N, n, mk)
    Q = np.sqrt(d) * M
    W = Q.T

    unitarity_residual = np.linalg.norm(W.conj().T @ W - np.eye(d))

    Phi_mat = np.eye(d) / np.sqrt(d)
    reconstructed_M = Phi_mat @ W.T

    reconstruction_residual = np.linalg.norm(reconstructed_M - M)

    return {"W": W, "unitarity_residual": unitarity_residual, "is_unitary": unitarity_residual <= atol, "reconstruction_residual": reconstruction_residual, "reconstructs_state": reconstruction_residual <= atol,}


# Shared reporting routine

def report_certificate(label: str, A: np.ndarray, m: int, k: int, result: Optional[Tuple[Sequence[int], Sequence[int]]], build_state: bool = True, max_mk_for_state: int = 6, state_override: Optional[np.ndarray] = None,) -> None:
    
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]

    print_parameters(m, k, A)
    print(f"Graph: {label}")

    if result is None:
        fig_path = plot_graph(A, title=label)
        print("  No valid balanced partition exists.")
        print("  Graph is NOT a valid encrypted cloning resource.")
        if fig_path:
            print(f"  Figure saved: {fig_path}")
        print()
        return

    S, N = result
    fig_path = plot_graph(A, S, N, title=label)
    print(f"  Certified! S = {set(S)}; N = {set(N)}")
    if fig_path:
        print(f"  Figure saved: {fig_path}")

    Gamma = cut_matrix(A, list(S), list(N))
    rank = gf2_rank(Gamma)
    print(f"  GF(2) rank(Gamma_S,N) = {rank}  (need {mk})  -> "
          f"{'PASS' if rank == mk else 'FAIL'}")

    if build_state and mk <= max_mk_for_state:
        psi = state_override if state_override is not None else graph_state_from_adjacency(A)
        rho_S = reduced_density_matrix(psi, list(S), n)
        matches, residual = rho_S_matches_maximally_mixed(rho_S, m, k)
        print(f"  rho_S == I/2^mk : {matches}   (||rho_S - I/2^mk||_F = {residual:.3e})")

        dec = construct_decoder(psi, S, N, m, k)
        print(f"  GSDC: W unitary : {dec['is_unitary']}   "
              f"(||W^dagger W - I||_F = {dec['unitarity_residual']:.3e})")
        print(f"  GSDC: |G> == (I_S ⊗ W)|Phi_{{{2 ** mk}}}> : "
              f"{dec['reconstructs_state']}   "
              f"(residual = {dec['reconstruction_residual']:.3e})")
    elif build_state:
        print(f"  (skipping explicit state construction: mk = {mk} > "
              f"max_mk_for_state = {max_mk_for_state}, state vector would "
              f"have 2^{n} = {2 ** n} amplitudes)")

    print()


def report_non_graph_state(label: str, psi: np.ndarray, n: int, m: int, k: int, S: Sequence[int], N: Sequence[int]) -> None:
    
    mk = m * k
    print_parameters(m, k, np.zeros((n, n), dtype=np.uint8))
    print(f"State: {label}  (S = {set(S)}, N = {set(N)})")
    rho_S = reduced_density_matrix(psi, list(S), n)
    matches, residual = rho_S_matches_maximally_mixed(rho_S, m, k)
    print(f"  rho_S == I/2^mk : {matches}   (||rho_S - I/2^mk||_F = {residual:.3e})  "
          f"-> {'valid encrypted-cloning resource' if matches else 'NOT a valid resource'}")

    dec = construct_decoder(psi, S, N, m, k)
    print(f"  GSDC: W unitary : {dec['is_unitary']}   "
          f"(||W^dagger W - I||_F = {dec['unitarity_residual']:.3e})")
    print()



In [42]:
print("=== GSECC certification + GSDC decoder construction ===\n")
print(f"Figures will be saved under: {FIGURE_DIR}\n")

# Common parameters used by the manuscript benchmark table:
# m = 2, k = 6  ->  2mk = 24 graph vertices.
m, k = 2, 6
n = 2 * m * k

# -------------------------------------------------------------------------
# Benchmark family names are kept exactly as in the manuscript table.
# -------------------------------------------------------------------------

# 1. Path / linear cluster
A = path_graph(m, k)
result = find_certificate_exact(A, m, k)
report_certificate("Path / linear cluster", A, m, k, result)

# 2. Cycle C_24
A = adjacency_matrix(n, [(i, (i + 1) % n) for i in range(n)])
result = find_certificate_exact(A, m, k)
report_certificate("Cycle C₂₄", A, m, k, result)

# 3. G^{cl}_{2 x 12}
rows, cols = 2, 12
A = grid_graph(rows, cols)
result = find_certificate_exact(A, m, k)
report_certificate("Gᶜˡ₂×₁₂", A, m, k, result)

# 4. Fixed G(24,0.35) sample
p = 0.35
rng = random.Random(7)
edges = [
    (i, j)
    for i in range(n)
    for j in range(i + 1, n)
    if rng.random() < p
]
A = adjacency_matrix(n, edges)
result = find_certificate_exact(A, m, k)
report_certificate(
    "Fixed G(24,0.35) sample",
    A, m, k, result,
    build_state=True,
    max_mk_for_state=6,
)

# 5. Fixed G(24,0.9) sample
p = 0.9
rng = random.Random(42)
edges = [
    (i, j)
    for i in range(n)
    for j in range(i + 1, n)
    if rng.random() < p
]
A = adjacency_matrix(n, edges)
result = find_certificate_exact(A, m, k)
report_certificate(
    "Fixed G(24,0.9) sample",
    A, m, k, result,
    build_state=False,
)

# 6. Complete graph K_24
A = adjacency_matrix(n, itertools.combinations(range(n), 2))
result = find_certificate_exact(A, m, k)
report_certificate("Complete graph K₂₄", A, m, k, result)

# 7. Star / GHZ graph
A = star_graph(m, k)
result = find_certificate_exact(A, m, k)
report_certificate("Star / GHZ graph", A, m, k, result)

# Parameter-validation sanity check retained separately from the graph-family list.
print("Deliberately mismatched (m, k) example:")
try:
    A_bad = adjacency_matrix(
        10,
        itertools.combinations(range(10), 2),
    )
    find_certificate_exact(A_bad, m, k)
except ValueError as e:
    print(f"  Rejected as expected: {e}")
print()

if os.path.isdir(FIGURE_DIR):
    saved = sorted(f for f in os.listdir(FIGURE_DIR) if f.endswith(".pdf"))
    print(f"Saved {len(saved)} figures (PDF + 600 dpi PNG each) to {FIGURE_DIR}:")
    for f in saved:
        print(f"  {f}")


=== GSECC certification + GSDC decoder construction ===

Figures will be saved under: C:\Users\roypr\Downloads\GESCC_temp

Encrypted Cloning Parameters
----------------------------
m  = 2
k  = 6
mk = 12
Graph vertices = 24
Graph: Path / linear cluster
  Certified! S = {0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22}; N = {1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23}
  Figure saved: C:\Users\roypr\Downloads\GESCC_temp\Path___linear_cluster.pdf
  GF(2) rank(Gamma_S,N) = 12  (need 12)  -> PASS
  (skipping explicit state construction: mk = 12 > max_mk_for_state = 6, state vector would have 2^24 = 16777216 amplitudes)

Encrypted Cloning Parameters
----------------------------
m  = 2
k  = 6
mk = 12
Graph vertices = 24
Graph: Cycle C₂₄
  Certified! S = {0, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21}; N = {2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 23}
  Figure saved: C:\Users\roypr\Downloads\GESCC_temp\Cycle_C₂₄.pdf
  GF(2) rank(Gamma_S,N) = 12  (need 12)  -> PASS
  (skipping explicit state construction: mk

### Sanity check for the new rank-deficient branch

This is the manuscript's \(m=3,\;k=1\) prescribed-cut example with

\[
A_S=A_{\mathcal N}=0,\qquad
B=
\begin{pmatrix}
1&0&0\\
0&1&0\\
1&1&0
\end{pmatrix},
\]

so \(\operatorname{rank}B=2\) and
\(\ker(B^{\mathsf T})=\operatorname{span}\{(1,1,1)^{\mathsf T}\}\).

The same fixed realization must pass for \(YZ/ZY\) and fail for \(XZ/ZX\).


In [43]:
# Small exact sanity check: rank-deficient prescribed realization
m_test, k_test = 3, 1
B_test = np.array(
    [
        [1, 0, 0],
        [0, 1, 0],
        [1, 1, 0],
    ],
    dtype=np.uint8,
)

A_test = np.zeros((6, 6), dtype=np.uint8)
A_test[:3, 3:] = B_test
A_test[3:, :3] = B_test.T

S_test = (0, 1, 2)
N_test = (3, 4, 5)
sectors_test = ((0, 1, 2),)

yz_result = verify_gsecc_realization(
    A_test, S_test, N_test, m_test, k_test,
    encoder="YZ",
    sectors=sectors_test,
)

xz_result = verify_gsecc_realization(
    A_test, S_test, N_test, m_test, k_test,
    encoder="XZ",
    sectors=sectors_test,
)

report_gsecc_result("Rank-deficient test: YZ/ZY", yz_result)
report_gsecc_result("Same realization: XZ/ZX", xz_result)

assert yz_result["valid"] is True
assert yz_result["rank"] == 2
assert yz_result["nullity"] == 1
assert xz_result["valid"] is False


Rank-deficient test: YZ/ZY
--------------------------
status        : valid
encoder class : YZ/ZY
branch         : rank_deficient_exceptional
S              : (0, 1, 2)
N              : (3, 4, 5)
rank(B)        : 2
nullity        : 1
kernel basis   : [(1, 1, 1)]
sectors        : ((0, 1, 2),)
reason         : Every basis vector of ker(B^T) lies in the encoder-selected exceptional subspace.

Same realization: XZ/ZX
-----------------------
status        : invalid
encoder class : XZ/ZX
branch         : rank_bound_failed
S              : (0, 1, 2)
N              : (3, 4, 5)
rank(B)        : 2
nullity        : 1
kernel basis   : [(1, 1, 1)]
sectors        : None
reason         : Necessary rank bound failed for XZ/ZX: rank=2, required rank >= 3.

